In [ ]:
%cd ../

# Import required libraries
from pathlib import Path
import hydra
from omegaconf import DictConfig, OmegaConf, open_dict
import torch
from lightning import seed_everything
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from distmap import euclidean_distance_transform
from modules.loader.loaders import trainScaler
from modules.transforms.tstransforms import TSResize
from cel.utils.reporting import sorted_inputs, sorted_inputs_regression
from modules.utils.seeds import seed_everything
from cel.utils import DSTMetric, RC, IC, soft_IC
from cel.utils.correctness import METRICS, dst_eclad
from cel.utils.misc import infer_shape
from os.path import isfile
import numpy as np
import json
import pandas as pd
from modules.dataset import HFDataset, npymapDataset
from datasets import Dataset
from modules.utils.hydraqol import run_decorator
from hydra.core.global_hydra import GlobalHydra

import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": 'serif',
    'font.serif': ['computer modern roman'],
    'font.size': 20
})

In [2]:
#@hydra.main(config_path="../data/config", config_name="train_model", version_base="1.3")
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(version_base=None, config_path="../data/config", job_name="test_app")

hydra.initialize()

In [ ]:
cfg = hydra.compose(config_name="CE_metrics.yaml", overrides=[
    "training=adamw_patient", "path=applewood_local",
    "model=DenseNet1D121", "dataset=midFreq-highFreq_ch1_specificChannels",
    "transforms=tsHFbasic", "loader=M", "seed=2",
    "CE.loader.batch_size=20", "CE.n_instances=200", "CE.n_clusters=50",
    "CE=CENDRE", "CE.n_concepts=0", "CE.clusterMode=ACsilhouette",
    "CE.maskMode=gradient", "CE.domain=[input,fourier]",
    "mode=force",
    "eval_metrics=['dst_eclad','soft_iou','soft_dice','cdc','epg','soft_precision','soft_recall','rma']",
])

In [ ]:
##############################
# Preliminaries
##############################
save_dir = Path(cfg.save_dir_model)

# Set the random seed for reproducibility
seed_everything(cfg.seed)

# Select the device for computation (CUDA, MPS, or CPU)
device = (
    "cuda"
    if (cfg.CE.device == "cuda" and torch.cuda.is_available())
    else "cpu"
)

##############################
# Object Instantiation
##############################
# Train scaler if necessary
if cfg.transforms.standardize:
    cfg.dataset.mean, cfg.dataset.std = trainScaler(cfg.dataset, cfg.loader)

In [ ]:
# Instantiate transforms for training and testing
pretransform = hydra.utils.instantiate(cfg.dataset.pretransform)
train_transform = hydra.utils.instantiate(cfg.transforms.train)
valid_transform = hydra.utils.instantiate(cfg.transforms.val)
test_transform = hydra.utils.instantiate(cfg.transforms.test)

# Instantiate datasets with transforms
train_dataset = hydra.utils.instantiate(cfg.dataset.train, transform=train_transform, pretransform=pretransform, withId=True)
valid_dataset = hydra.utils.instantiate(cfg.dataset.val, transform=valid_transform, pretransform=pretransform, withId=True)
test_dataset = hydra.utils.instantiate(cfg.dataset.test, transform=test_transform, pretransform=pretransform, withId=True)

# Instantiate data loaders
train_loader, valid_loader, _ = hydra.utils.instantiate(
    cfg.CE.loader,
    dataset_train=train_dataset,
    dataset_val=valid_dataset,
    dataset_test=test_dataset,
    task=cfg.dataset.task,
)

In [ ]:
# Instantiate model
model = hydra.utils.instantiate(cfg.model.object)
PATH = f"{save_dir}/best.pth"
model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()

In [ ]:
# Initialize CE method
CE = hydra.utils.instantiate(cfg.CE.object, model=model)

In [ ]:
###############################################
# CE Training Loop (Fit CE if appropriate)
###############################################
fitExists, reportExists = CE.check_existence(cfg.CE)
if cfg.mode == "force" or not fitExists:
    print("training Explainer")
    CE.fit(train_loader, cfg.CE, device=device)
    CE.save(cfg.CE)
    with open(f"{cfg.CE.save_dir}/config.yaml", "w") as f:
        OmegaConf.save(config=cfg, f=f)
else:
    CE.load(cfg.CE, device)
    print("loaded Explainer from files")

In [ ]:
# Make CE report if it does not exist already
if  cfg.mode == "force" or cfg.mode == "force-report" or not reportExists:
    if cfg.dataset.task == "classification":
        batch = sorted_inputs(
            CE.model, valid_loader, n_classes=cfg.dataset.n_classes, n_repeats=2
        )
    elif cfg.dataset.task == "regression" and cfg.transforms.standardize:
        batch = sorted_inputs_regression(
            valid_loader, partitions=5,samples_per_partition=1, out_range=[-2,2],int_out=cfg.CE.multilabel_out, astensor=True
        )
    else:
        batch = next(iter(valid_loader))

    if isinstance(batch, dict):
        test_inputs, test_truth, test_ids = batch["x"][:4], batch["y"][:4], batch["id"][:4]
    else:
        test_inputs, test_truth, test_ids = batch

    for domain in cfg.CE.domain:
        print(f'Plotting report on {domain} domain')
        test_masks, test_outputs = CE.eval(test_inputs, domain=domain, device=device, maskMode=getattr(cfg.CE, 'maskMode', None)) 

        CE.report(
            test_masks,
            cfg.CE,
            test_inputs,
            test_outputs,
            domain=domain,
            test_truth=test_truth,
            DS_cfg=cfg.dataset,
            font_base = 5,
            destandardize=cfg.transforms.standardize,
            visualization_mode=getattr(cfg.CE, 'visualization_mode', None),
        )

        plt.show()

In [ ]:
# Plot primitives for the same instances as the report
input_shape = infer_shape(valid_loader)
input_len = input_shape[1:]
n_channels = input_shape[0] if cfg.CE.decorrelated else 1

if len(input_len) == 1:
    resize = TSResize(new_length=torch.tensor(input_len), mode="nearest")
else:
    raise NotImplementedError

# Load primitives
report_primitives = {}
if isinstance(train_dataset, HFDataset):
    ds = Dataset.load_from_disk(cfg.dataset.val.dataset.dataset_path)
    for p, pname in enumerate(cfg.dataset.maskcols):
        report_primitives[p] = torch.tensor(ds[pname], dtype=torch.float32)
elif isinstance(train_dataset.dataset, npymapDataset):
    pathlist = sorted(Path(cfg.dataset.eval_path).glob('**/*.npy'))
    for p, path in enumerate(pathlist):
        report_primitives[p] = torch.tensor(np.load(path), dtype=torch.float32)

# Expand single-channel primitives to multi-channel if needed
if n_channels > 1:
    xcols = list(getattr(cfg.dataset, 'xcols', []))
    def _expand_masks(masks, maskcol_names):
        for p, pname in enumerate(maskcol_names):
            ch_idx = -1
            for ci, xc in enumerate(xcols):
                if pname.endswith(f'_{xc}'):
                    ch_idx = ci
                    break
            if ch_idx >= 0:
                raw = masks[p]
                if raw.dim() == 2:
                    expanded = torch.zeros(raw.shape[0], n_channels, raw.shape[-1], dtype=raw.dtype)
                    expanded[:, ch_idx, :] = raw
                elif raw.dim() == 3 and raw.shape[1] == 1:
                    expanded = torch.zeros(raw.shape[0], n_channels, raw.shape[-1], dtype=raw.dtype)
                    expanded[:, ch_idx, :] = raw[:, 0, :]
                else:
                    continue
                masks[p] = expanded
    _expand_masks(report_primitives, list(cfg.dataset.maskcols))

# Select the same instances and resize
n_primitives = len(report_primitives)
batch_size = test_inputs.shape[0]
maskcol_names = list(cfg.dataset.maskcols) if hasattr(cfg.dataset, 'maskcols') else [f'P{p}' for p in range(n_primitives)]
test_ids_long = test_ids.long()

# Compute x_scale (same as report)
domain = cfg.CE.domain[0]
fs = getattr(cfg.dataset, 'fs', 1)
T = test_inputs.shape[-1]
if domain == 'fourier':
    x_scale = fs * np.arange(0, T) / (2 * T)
else:
    x_scale = np.arange(0, T) / fs

fig, axes = plt.subplots(n_primitives, batch_size, figsize=(4 * batch_size, 2 * n_primitives), squeeze=False)

for p in range(n_primitives):
    prim_batch = resize(report_primitives[p][test_ids_long])  # (batch, [ch], T)
    for i in range(batch_size):
        ax = axes[p, i]
        if prim_batch.dim() == 3:
            for ch in range(prim_batch.shape[1]):
                ax.plot(x_scale, prim_batch[i, ch, :].numpy(), linewidth=1)
        else:
            ax.plot(x_scale, prim_batch[i].numpy(), linewidth=1)
        ax.set_xlim(x_scale[0], x_scale[-1])
        if i == 0:
            ax.set_ylabel(maskcol_names[p], fontsize=10, rotation=90)
        else:
            ax.set_yticklabels([])
        if p < n_primitives - 1:
            ax.set_xticklabels([])

plt.suptitle('Primitives (aligned with report instances)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
###################################
# Actual Task: Metric computation
###################################

# Load primitives and confounders
primitives = {}
confounders = {}
if isinstance(train_dataset, HFDataset):
    ds = Dataset.load_from_disk(cfg.dataset.val.dataset.dataset_path)
    for p, pname in enumerate(cfg.dataset.maskcols):
        primitives[p] = torch.tensor(ds[pname], dtype=torch.float32)
    for p, pname in enumerate(cfg.dataset.counfoundercols):
        confounders[p] = torch.tensor(ds[pname], dtype=torch.float32)

elif isinstance(train_dataset.dataset, npymapDataset):
    pathlist = sorted(Path(cfg.dataset.eval_path).glob('**/*.npy'))
    for p, path in enumerate(pathlist):
        primitives[p] = torch.tensor(np.load(path), dtype=torch.float32)

# Expand single-channel primitives to multi-channel if needed
if n_channels > 1:
    xcols = list(getattr(cfg.dataset, 'xcols', []))
    def _expand_masks(masks, maskcol_names):
        for p, pname in enumerate(maskcol_names):
            ch_idx = -1
            for ci, xc in enumerate(xcols):
                if pname.endswith(f'_{xc}'):
                    ch_idx = ci
                    break
            if ch_idx >= 0:
                raw = masks[p]
                if raw.dim() == 2:
                    expanded = torch.zeros(raw.shape[0], n_channels, raw.shape[-1], dtype=raw.dtype)
                    expanded[:, ch_idx, :] = raw
                elif raw.dim() == 3 and raw.shape[1] == 1:
                    expanded = torch.zeros(raw.shape[0], n_channels, raw.shape[-1], dtype=raw.dtype)
                    expanded[:, ch_idx, :] = raw[:, 0, :]
                else:
                    continue
                masks[p] = expanded
    _expand_masks(primitives, list(cfg.dataset.maskcols))
    if confounders:
        _expand_masks(confounders, list(cfg.dataset.counfoundercols))

In [ ]:
# --- Setup for DSTMetric ---
concept_ids = list(CE.importances.keys())
n_concepts = len(concept_ids)

has_confounders = bool(getattr(cfg.dataset, 'counfoundercols', []))
suffixes = ['']
if has_confounders:
    suffixes.append('_conf')

# --- Lightning loggers ---
tb_logger = TensorBoardLogger(save_dir=cfg.CE.save_dir, name="metrics")
csv_logger = CSVLogger(save_dir=cfg.CE.save_dir, name="metrics")

# Compute metrics for each configured eval metric
print('computing metrics')
metrics = {}
eval_metrics = list(cfg.eval_metrics) if hasattr(cfg, 'eval_metrics') else ['dst_eclad']
mask_sets = [('', primitives)]
if confounders:
    mask_sets.append(('_conf', confounders))

# --- Precompute max_penalties and create all DSTMetric instances ---
extreme = torch.ones(input_len, device=device)
extreme[0] = 0
dst_max_penalty = float(torch.sum(euclidean_distance_transform(extreme)) / (max(input_len) ** 2))

max_penalties = {}
dst_metrics = {}
for metric_name in eval_metrics:
    metric_info = METRICS[metric_name]
    max_penalties[metric_name] = dst_max_penalty if metric_info["fn"] is dst_eclad else 1.0
    for suffix, masks in mask_sets:
        dst_metrics[(metric_name, suffix)] = DSTMetric(
            n_primitives=len(masks), n_concepts=n_concepts,
            n_channels=n_channels, metric_fn=metric_info["fn"],
            max_penalty=max_penalties[metric_name],
        ).to(device)

# Group by maskMode to call CE.eval once per unique maskMode per batch
maskModes = {METRICS[m]["maskMode"] for m in eval_metrics}

# --- Dataloader loop (outer) ---
for i, data in enumerate(valid_loader):
    if isinstance(data, dict):
        x, ids = data["x"], data["id"]
    else:
        x, _, ids = data

    # Eval once per unique maskMode
    eval_by_maskMode = {}
    for maskMode in maskModes:
        eval_by_maskMode[maskMode], _ = CE.eval(x, device, maskMode=maskMode, domain=cfg.CE.domain[0])

    # Resize concept masks once per maskMode
    concept_dicts = {
        mm: {c: resize(eval_masks[int(d)]) for c, d in enumerate(concept_ids)}
        for mm, eval_masks in eval_by_maskMode.items()
    }
    # Resize primitive masks once per mask set
    prim_dicts = {
        suffix: {p: resize(masks[p][ids]) for p in range(len(masks))}
        for suffix, masks in mask_sets
    }

    # Update all metric instances (inner loop)
    for metric_name in eval_metrics:
        maskMode = METRICS[metric_name]["maskMode"]
        for suffix, _ in mask_sets:
            dst_metrics[(metric_name, suffix)].update(
                prim_dicts[suffix], concept_dicts[maskMode], n_channels
            )

    if (i + 1) * cfg.CE.loader.batch_size >= cfg.CE.n_instances:
        break

# --- Compute results, store alignments, and log ---
import logging
all_alignments = {}
all_sim = {}
for metric_name in eval_metrics:
    max_penalty = max_penalties[metric_name]
    for suffix, _ in mask_sets:
        scores = dst_metrics[(metric_name, suffix)].compute()
        np.save(
            f'{cfg.CE.save_dir}/scores_{metric_name}{suffix}_{str(cfg.CE.n_concepts)}.npy',
            scores.detach().cpu().numpy()
        )
        metrics[f'RC_{metric_name}{suffix}'], alignments, sim = RC(
            scores, 'rel', max_penalty, device=device,
            higher_is_better=METRICS[metric_name]["higher_is_better"]
        )
        all_alignments[(metric_name, suffix)] = alignments
        all_sim[(metric_name, suffix)] = sim

        try:
            metrics[f'IC_{metric_name}{suffix}'], metrics[f'ICA_{metric_name}{suffix}'], metrics[f'ICU_{metric_name}{suffix}'] = IC(
                alignments, CE, device=device
            )
        except Exception as e:
            logging.warning("IC failed for %s%s: %s", metric_name, suffix, e)
            metrics[f'IC_{metric_name}{suffix}'] = float('nan')
            metrics[f'ICA_{metric_name}{suffix}'] = float('nan')
            metrics[f'ICU_{metric_name}{suffix}'] = float('nan')

        try:
            metrics[f'sIC_{metric_name}{suffix}'] = soft_IC(
                sim, CE, device=device
            )
        except Exception as e:
            logging.warning("soft_IC failed for %s%s: %s", metric_name, suffix, e)
            metrics[f'sIC_{metric_name}{suffix}'] = float('nan')

# Log via Lightning loggers
for logger in [tb_logger, csv_logger]:
    logger.log_metrics(metrics, step=0)
    logger.save()
    logger.finalize("success")

# Backward-compatible CSV (same filename and format)
df = pd.DataFrame(metrics, index=[0])
df.to_csv(f'{cfg.CE.save_dir}/metrics_{str(cfg.CE.n_concepts)}.csv')
for metric_name in eval_metrics:
    print(f"  {metric_name}: RC={metrics.get(f'RC_{metric_name}', float('nan')):.4f}, IC={metrics.get(f'IC_{metric_name}', float('nan')):.4f}, ICA={metrics.get(f'ICA_{metric_name}', float('nan')):.4f}, ICU={metrics.get(f'ICU_{metric_name}', float('nan')):.4f}, sIC={metrics.get(f'sIC_{metric_name}', float('nan')):.4f}")

In [ ]:
# --- Primitive alignment info (similar to task32) ---
prim_names = list(cfg.dataset.maskcols) if hasattr(cfg.dataset, 'maskcols') else [f'prim_{p}' for p in range(len(primitives))]
imp_display = {int(k): (max(v) if isinstance(v, list) else float(v)) for k, v in CE.importances.items()}

for metric_name in eval_metrics:
    for suffix in suffixes:
        if (metric_name, suffix) not in all_alignments:
            continue
        alignments = all_alignments[(metric_name, suffix)]
        sim = all_sim[(metric_name, suffix)]
        n_p, n_c, n_ch = alignments.shape

        rc_val = metrics.get(f'RC_{metric_name}{suffix}', float('nan'))
        ic_val = metrics.get(f'IC_{metric_name}{suffix}', float('nan'))
        suffix_label = f" (confounders)" if suffix == '_conf' else ""
        print(f"\n{'='*60}")
        print(f"{metric_name}{suffix_label}  —  RC={rc_val:.4f}, IC={ic_val:.4f}")
        print(f"{'='*60}")

        aligned_concepts = set()
        for p in range(n_p):
            pname = prim_names[p] if p < len(prim_names) else f"prim_{p}"
            aligned = []
            for c in range(n_c):
                for ch in range(n_ch):
                    if alignments[p, c, ch] > 0:
                        aligned_concepts.add(c)
                        ch_str = f"/ch{ch}" if n_ch > 1 else ""
                        aligned.append(f"C{c}{ch_str} (sim={sim[p, c, ch]:.3f})")
            aligned_str = ", ".join(aligned) if aligned else "—"
            print(f"  {pname:30s} -> {aligned_str}")

        unaligned = []
        for c in range(n_c):
            if c not in aligned_concepts:
                imp = imp_display.get(c, 0.0)
                unaligned.append(f"C{c} (imp={imp:.3f})")
        if unaligned:
            print(f"  Unaligned concepts: {', '.join(unaligned)}")

In [ ]:
# --- Nearest concept per primitive (across metrics) ---
prim_names = list(cfg.dataset.maskcols) if hasattr(cfg.dataset, 'maskcols') else [f'prim_{p}' for p in range(len(primitives))]

# Build scores dict: load from the just-saved .npy files (main suffix only)
scores_by_metric = {}
for metric_name in eval_metrics:
    path = f'{cfg.CE.save_dir}/scores_{metric_name}_{str(cfg.CE.n_concepts)}.npy'
    if isfile(path):
        scores_by_metric[metric_name] = torch.tensor(np.load(path), dtype=torch.float32)

# Build a DataFrame: rows = primitives, columns = metrics, cells = "score (C#)"
rows = []
for p, pname in enumerate(prim_names):
    row = {'Primitive': pname}
    for metric_name, scores in scores_by_metric.items():
        # scores shape: (n_primitives, n_concepts, n_channels)
        higher = METRICS[metric_name]["higher_is_better"]
        # For each primitive, find the best concept (aggregate across channels)
        prim_scores = scores[p]  # (n_concepts, n_channels)
        if higher:
            best_val, best_idx = prim_scores.amax(dim=1).max(dim=0)
            best_ch = prim_scores[best_idx].argmax().item()
        else:
            best_val, best_idx = prim_scores.amin(dim=1).min(dim=0)
            best_ch = prim_scores[best_idx].argmin().item()
        ch_str = f"/ch{best_ch}" if prim_scores.shape[1] > 1 else ""
        row[metric_name] = f"{best_val.item():.4f} (C{best_idx.item()}{ch_str})"
    rows.append(row)

df_nearest = pd.DataFrame(rows).set_index('Primitive')
print("\nNearest Concept per Primitive:")
print(df_nearest.to_string())

In [ ]:
# Display results
metric = pd.read_csv(f'{cfg.CE.save_dir}/metrics_{str(cfg.CE.n_concepts)}.csv')
metric